# 5-Class Emergency Department Triage Classifier with 5-Fold Group Cross-Validation & DeLong AUROC Confidence Intervals (`models/train_fedmm_classifier.ipynb`)

This notebook implements patient-level **5-Fold Stratified Group Cross-Validation** for **5-Class Emergency Severity Index (ESI 1..5) LightGBM triage models** on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)**:

### 🔬 Key Methodological Innovations
1. **Encounter-Level 5-Fold Cross-Validation with Patient-Level Isolation**:
   - Uses `sklearn.model_selection.StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` grouped on `patient_id`.
   - **Strict Guarantee**: Ensures patients in the training set **never appear in the test set** (zero patient overlap across all 5 folds), preventing clustered encounter leakage.
2. **Leakage-Free MICE Imputation**:
   - Chained equation imputation (`IterativeImputer`) is **fit exclusively on the training encounters of each fold** and applied to transform the fold's test encounters.
3. **DeLong's Non-Parametric Method for AUROC Confidence Intervals**:
   - Computes exact asymptotic standard errors (SE) and **95% Wald Confidence Intervals** for One-vs-Rest ROC-AUC curves using DeLong's U-statistic structural components algorithm:
     $$\text{CI}_{95\%} = \Big[ \max\big(0, \, \widehat{\text{AUC}} - 1.96 \cdot \text{SE}_{\text{DeLong}}\big), \; \min\big(1, \, \widehat{\text{AUC}} + 1.96 \cdot \text{SE}_{\text{DeLong}}\big) \Big]$$

### 📊 Feature Roster & Target Specification
- **Predictor Features (6 Features)**:
  1. `age`: Patient age in years (numerical continuous).
  2. `sex`: Biological sex encoded as integer (`1 = Male ('M')`, `0 = Female ('F')`).
  3. `systolic_bp`: Systolic Blood Pressure in mmHg (continuous, MICE imputed).
  4. `heart_rate`: Heart Rate in bpm (continuous, MICE imputed).
  5. `respiratory_rate`: Respiratory Rate in breaths/min (continuous, MICE imputed).
  6. `spo2`: Blood Oxygen Saturation in % (continuous, MICE imputed).
- **Target Class**:
  - `esi_level`: Discrete 5-class triage score ($1..5$).

```mermaid
flowchart TD
    RawData["FedMML Dataset (87,234 encounters, 96 patients)"] --> SGKF["5-Fold Stratified Group K-Fold (grouped by patient_id)"]
    
    subgraph FoldLoop ["5-Fold Cross-Validation Loop (Fold k = 1..5)"]
        SplitK["Split Fold k: Train Encounters (distinct patients) & Test Encounters (unseen patients)"]
        MICE_K["MICE Imputation (fit on Train Fold, transform Test Fold)"]
        Scale_K["StandardScaler Normalization"]
        Train_K["Train LightGBM (objective='multiclass', class_weight='balanced')"]
        Predict_K["Out-of-Fold (OOF) Test Probability Predictions"]
        SplitK --> MICE_K --> Scale_K --> Train_K --> Predict_K
    end
    
    SGKF --> FoldLoop
    Predict_K --> OOF_Aggregate["Aggregate Full Dataset OOF Predictions & Probabilities (N = 87,234)"]
    OOF_Aggregate --> DeLong["DeLong AUROC Standard Errors & 95% Confidence Intervals"]
    DeLong --> Eval["Comprehensive Report, Confusion Matrix, ROC-AUC Curves with 95% CI"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Encode 'sex', & Prepare Patient Groups
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
df = pd.read_csv(data_path)
total_encounters = len(df)
unique_patients  = df['patient_id'].nunique()

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'
group_col  = 'patient_id'

# 2. Encode 'sex' Feature (M -> 1, F -> 0)
df['sex_encoded'] = df['sex'].astype(str).str.strip().str.upper().map({'M': 1.0, 'MALE': 1.0, 'F': 0.0, 'FEMALE': 0.0})
feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

df = df.dropna(subset=[target_col]).copy()
df[target_col] = df[target_col].astype(int)

print("=" * 85)
print(f"  FEDMML DATASET: {total_encounters:,} Encounters across {unique_patients:,} Unique Patients")
print("=" * 85)
print("Missing Value Count per Feature (to be imputed using MICE inside each fold):")
print(df[feature_names].isnull().sum())
print("-" * 85)
print("Target ESI Distribution:")
esi_dist = df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} encounters ({cnt/len(df)*100:.2f}%)")
print("=" * 85)

In [ ]:
# ---------------------------------------------------------
# Step 2: Vectorized DeLong Method for AUROC Variance & 95% Confidence Intervals
# Reference: DeLong et al. (1988), Biometrics 44(3):837-845
# ---------------------------------------------------------
def delong_roc_variance(ground_truth_binary, predictions_continuous, alpha=0.05):
    """
    Computes exact AUROC, DeLong asymptotic standard error (SE), and (1 - alpha)% confidence interval.
    Vectorized O(N log N) algorithm using sorted structural midranks.
    """
    pos = predictions_continuous[ground_truth_binary == 1]
    neg = predictions_continuous[ground_truth_binary == 0]
    m = len(pos)
    n = len(neg)
    
    if m == 0 or n == 0:
        return 0.0, 0.0, (0.0, 0.0), "[0.0000 - 0.0000]"
    
    pos_sorted = np.sort(pos)
    neg_sorted = np.sort(neg)
    
    # Structural components V10 and V01 via fast binary search
    v10 = (np.searchsorted(neg_sorted, pos, side='left') + np.searchsorted(neg_sorted, pos, side='right')) / (2.0 * n)
    v01 = 1.0 - (np.searchsorted(pos_sorted, neg, side='left') + np.searchsorted(pos_sorted, neg, side='right')) / (2.0 * m)
    
    auc = float(np.mean(v10))
    s10 = float(np.var(v10, ddof=1)) if m > 1 else 0.0
    s01 = float(np.var(v01, ddof=1)) if n > 1 else 0.0
    
    variance = (s10 / m) + (s01 / n)
    se = float(np.sqrt(max(0.0, variance)))
    
    z_crit = 1.959963984540054  # 95% Normal critical value
    ci_lower = max(0.0, auc - z_crit * se)
    ci_upper = min(1.0, auc + z_crit * se)
    ci_str = f"[{ci_lower:.4f} - {ci_upper:.4f}]"
    
    return auc, se, (ci_lower, ci_upper), ci_str

print("✓ Vectorized DeLong's Method for AUROC Variance & 95% CI loaded successfully!")

In [ ]:
# ---------------------------------------------------------
# Step 3: 5-Fold Stratified Group Cross-Validation Training Loop
# ---------------------------------------------------------
print("=" * 90)
print("  5-FOLD STRATIFIED GROUP CROSS-VALIDATION (ISOLATING PATIENTS ACROSS FOLDS)")
print("=" * 90)

X_raw  = df[feature_names].values
y_true = df[target_col].values
groups = df[group_col].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold probability matrix and prediction buffer
oof_probs = np.zeros((len(df), 5), dtype=np.float64)
oof_preds = np.zeros(len(df), dtype=int)
fold_models  = []
fold_scalers = []
fold_imputers = []

cont_indices = [0, 2, 3, 4, 5]  # age, systolic_bp, heart_rate, respiratory_rate, spo2

lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'class_weight': 'balanced',
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'min_child_samples': 20,
    'n_estimators': 200,
    'verbosity': -1,
    'random_state': 42
}

t_start = time.time()

for fold, (tr_idx, test_idx) in enumerate(sgkf.split(X_raw, y_true, groups)):
    t_fold = time.time()
    
    # 1. Verify Patient Isolation
    pats_tr   = set(groups[tr_idx])
    pats_test = set(groups[test_idx])
    overlap   = pats_tr.intersection(pats_test)
    assert len(overlap) == 0, f"Fatal: Patient overlap detected in Fold {fold+1}!"
    
    # 2. Extract Raw Fold Data
    X_tr_raw_f   = X_raw[tr_idx]
    y_tr_f       = y_true[tr_idx]
    groups_tr_f  = groups[tr_idx]
    
    X_test_raw_f = X_raw[test_idx]
    y_test_f     = y_true[test_idx]
    
    # Internal validation split from training fold for early stopping (patient-grouped)
    sgkf_inner = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42 + fold)
    in_tr_idx, in_val_idx = next(sgkf_inner.split(X_tr_raw_f, y_tr_f, groups_tr_f))
    
    # 3. Fit MICE Imputer exclusively on Training encounters
    imputer_fold = IterativeImputer(max_iter=10, random_state=42 + fold, verbose=0)
    X_in_tr_imp   = imputer_fold.fit_transform(X_tr_raw_f[in_tr_idx])
    X_in_val_imp  = imputer_fold.transform(X_tr_raw_f[in_val_idx])
    X_test_imp_f  = imputer_fold.transform(X_test_raw_f)
    
    # 4. Standardize continuous features
    scaler_fold = StandardScaler()
    X_in_tr_s   = X_in_tr_imp.copy()
    X_in_val_s  = X_in_val_imp.copy()
    X_test_s    = X_test_imp_f.copy()
    
    X_in_tr_s[:, cont_indices]  = scaler_fold.fit_transform(X_in_tr_imp[:, cont_indices])
    X_in_val_s[:, cont_indices] = scaler_fold.transform(X_in_val_imp[:, cont_indices])
    X_test_s[:, cont_indices]   = scaler_fold.transform(X_test_imp_f[:, cont_indices])
    
    # 5. Train LightGBM model
    model_fold = lgb.LGBMClassifier(**lgb_params)
    model_fold.fit(
        X_in_tr_s, y_tr_f[in_tr_idx] - 1,
        eval_set=[(X_in_val_s, y_tr_f[in_val_idx] - 1)],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    # 6. Out-of-fold predictions on Holdout Fold (strictly unseen patients)
    test_probs = model_fold.predict_proba(X_test_s)
    test_preds = np.argmax(test_probs, axis=1) + 1
    
    oof_probs[test_idx] = test_probs
    oof_preds[test_idx] = test_preds
    
    fold_bal_acc = balanced_accuracy_score(y_test_f, test_preds)
    fold_models.append(model_fold)
    fold_scalers.append(scaler_fold)
    fold_imputers.append(imputer_fold)
    
    print(f"✓ Fold {fold+1}/5 Complete ({time.time()-t_fold:.1f}s): Train Encounters={len(tr_idx):,} ({len(pats_tr)} pats), "
          f"Test Encounters={len(test_idx):,} ({len(pats_test)} pats), Overlap=0 | Fold Bal Acc: {fold_bal_acc*100:.2f}%")

print("=" * 90)
print(f"✓ 5-Fold Group Cross-Validation finished in {time.time()-t_start:.1f}s!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Full Out-Of-Fold Benchmark Evaluation & DeLong 95% Confidence Intervals
# ---------------------------------------------------------
def compute_comprehensive_delong_metrics(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs, ses = [], [], [], [], [], [], []
    
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        auc, se, ci, ci_str = delong_roc_variance(y_bin_true, probs[:, idx])
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc); ses.append(se)
        
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'AUROC': round(auc, 4),
            'DeLong_SE': round(se, 4),
            'DeLong_95_CI': ci_str
        })
    
    # Macro-average with pooled DeLong standard error
    macro_auc = float(np.mean(aucs))
    macro_se  = float(np.sqrt(np.sum(np.array(ses)**2)) / len(classes))
    macro_ci_low = max(0.0, macro_auc - 1.96 * macro_se)
    macro_ci_up  = min(1.0, macro_auc + 1.96 * macro_se)
    macro_ci_str = f"[{macro_ci_low:.4f} - {macro_ci_up:.4f}]"
    
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'AUROC': round(macro_auc, 4),
        'DeLong_SE': round(macro_se, 4),
        'DeLong_95_CI': macro_ci_str
    })
    return pd.DataFrame(rows)

report_delong_df = compute_comprehensive_delong_metrics(
    y_true, oof_preds, oof_probs, 'FedMML_5Fold_GroupCV_LightGBM_MICE'
)

print("=" * 125)
print("   OUT-OF-FOLD (OOF) 5-FOLD GROUP-CV EVALUATION WITH DELONG AUROC 95% CONFIDENCE INTERVALS (N = 87,234)")
print("=" * 125)
print(report_delong_df[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'AUROC', 'DeLong_SE', 'DeLong_95_CI']].to_string(index=False))
print("=" * 125 + chr(10))

# Export CSV Report
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)
report_path = os.path.join(reports_dir, 'fedmml_5fold_groupcv_delong_report.csv')
report_delong_df.to_csv(report_path, index=False)
print(f"✓ Performance report with DeLong 95% CIs saved to: {report_path}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Diagnostic Visualizations (OOF Confusion Matrix, ROC-AUC Curves with DeLong CIs & Feature Importance)
# ---------------------------------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. Out-of-Fold 5x5 Normalized Confusion Matrix
cm      = confusion_matrix(y_true, oof_preds, labels=[1, 2, 3, 4, 5])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

annot = np.empty_like(cm, dtype=object)
for i in range(5):
    for j in range(5):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
macro_bal = report_delong_df.loc[5, 'Balanced_Accuracy']
macro_auc_val = report_delong_df.loc[5, 'AUROC']
macro_ci_text = report_delong_df.loc[5, 'DeLong_95_CI']

ax.set_title(
    f"FedMML 5-Fold Patient Group-CV Confusion Matrix (N = 87,234)\n"
    f"Macro Balanced Acc: {macro_bal*100:.2f}% | Macro AUC: {macro_auc_val:.4f} {macro_ci_text}",
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
ax.set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, "fedmml_5fold_groupcv_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_5fold_groupcv_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 5-Fold Group-CV Confusion Matrix saved to: {cm_path}")

# 2. Multiclass One-vs-Rest ROC-AUC Curves with DeLong 95% Confidence Intervals
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
y_all_bin = label_binarize(y_true, classes=classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fpr, tpr, roc_aucs = dict(), dict(), dict()
for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_all_bin[:, i], oof_probs[:, i])
    roc_aucs[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_all_bin.ravel(), oof_probs.ravel())
roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9.5, 8))
plt.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {macro_auc_val:.4f}, 95% CI {macro_ci_text})", color='#17becf', linestyle='--', linewidth=2.5)

for i in range(5):
    cls_ci = report_delong_df.loc[i, 'DeLong_95_CI']
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f}, 95% CI {cls_ci})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
plt.title(f"FedMML 5-Fold Patient Group-CV Multiclass ROC-AUC Curves\n(With DeLong 95% Confidence Intervals)", fontsize=12, fontweight='bold', pad=10)
plt.legend(loc="lower right", fontsize=9.2, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
roc_path = os.path.join(plots_dir, "fedmml_5fold_groupcv_roc_auc_curve.png")
plt.savefig(roc_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_5fold_groupcv_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ ROC-AUC Curves with DeLong CIs saved to: {roc_path}")

# 3. Mean Feature Importance across all 5 Folds
mean_gains = np.mean([m.booster_.feature_importance(importance_type='gain') for m in fold_models], axis=0)
mean_splits = np.mean([m.booster_.feature_importance(importance_type='split') for m in fold_models], axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Importance_Gain': mean_gains,
    'Mean_Importance_Split': mean_splits
}).sort_values('Mean_Importance_Gain', ascending=False)

sns.barplot(data=feat_imp, x='Mean_Importance_Gain', y='Feature', palette='Blues_r', ax=ax, edgecolor='black')
ax.set_title('5-Fold Mean LightGBM Feature Importance (Information Gain)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Mean Total Information Gain', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature Name', fontsize=11, fontweight='bold')

for p in ax.patches:
    ax.annotate(f"{p.get_width():,.1f}", (p.get_width(), p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=9.5, fontweight='bold', xytext=(5, 0), textcoords='offset points')

plt.tight_layout()
feat_imp_path = os.path.join(plots_dir, "fedmml_5fold_groupcv_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_5fold_groupcv_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Mean Feature Importance plot saved to: {feat_imp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 6: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'fold_models': fold_models,
    'fold_scalers': fold_scalers,
    'fold_imputers': fold_imputers,
    'feature_names': feature_names,
    'cv_strategy': '5-Fold StratifiedGroupKFold on patient_id'
}

bundle_file = os.path.join(deploy_dir, 'fedmml_5fold_groupcv_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_5Fold_Patient_GroupCV_LightGBM_Classifier',
    dataset='datasets/fedmml_ed_triage_dataset.csv',
    cross_validation='StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)',
    group_variable='patient_id',
    patient_isolation='Zero patient overlap between training and test folds',
    imputation_method='MICE_IterativeImputer (fit inside each training fold)',
    auroc_confidence_intervals='DeLong non-parametric U-statistic method (95% Wald CI)',
    total_encounters=len(df),
    unique_patients=unique_patients,
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    target='esi_level',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    oof_evaluation_metrics=report_delong_df.to_dict(orient='records')
)

manifest_file = os.path.join(deploy_dir, 'fedmml_5fold_groupcv_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")